# **Wooclap Data Case Study**

**Objectives:**
1. Create a script to write the CSV dataset in parquet format using a partioning per day
2. Make a graph to analyze the distribution of answers over time
3. Propose a method to identify sessions

In [92]:
#import the necessary libaries before the session
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import os
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [72]:
#import the data and analyze what we have so far
df = pd.read_csv('sample.csv')
df

,Unnamed: 0,type,participant_id,event_id,created_at,object_id
0,1,question-answer,5bb21a6099567277d5856835,5bae4c0addbff83a25cef9e4,2018-10-04T09:03:57Z,5bb5bca6016c832eacc6a99f
1,2,question-answer,5b99018865b984370bf79d24,5b86edb9b60e0a2cb4447e78,2018-10-05T07:01:06Z,5b92b11c050fd04b90581398
2,3,question-answer,z112758121313,5b9e75a9416fae3713ad2eca,2018-10-09T12:26:25Z,5ba0d22631c09e2da13f27f8
3,4,question-answer,z211158485667,5ba8c5a723306e3a1808e753,2018-09-25T08:33:01Z,5ba8c6ebd0f1ca3a1e4b6e33
4,5,question-answer,5b96c5d5e3e3334b8a1f4fc1,5b9681ad52d9f12c9f277095,2018-09-10T19:44:14Z,5b96997e0f570e2c9e0a15c3
...,...,...,...,...,...,...
10095,10096,question-answer,z261997914177,5bd6d4d19463065f1402e454,2018-11-15T15:53:25.163Z,5bd6d4d19463065f1402e517
10096,10097,question-answer,z429552861175,5bc397b55ea8aa5f07340e14,2018-10-15T12:10:25.315Z,5bc4534019778f2cce061e35
10097,10098,question-answer,z520001780226,5c025a203c6d2757c232e3ae,2018-12-07T10:33:14.823Z,5c025a203c6d2757c232e3e3
10098,10099,question-answer,z255132820634,5b5736717d33bf57dd73434b,2018-09-19T09:18:35.797Z,5b5736717d33bf57dd734378


In [73]:
#check to see what kind of data is stored in this dataframe (object,integer,etc)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10100 entries, 0 to 10099
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Unnamed: 0      10100 non-null  int64 
 1   type            10100 non-null  object
 2   participant_id  10100 non-null  object
 3   event_id        10100 non-null  object
 4   created_at      10100 non-null  object
 5   object_id       10100 non-null  object
dtypes: int64(1), object(5)
memory usage: 473.6+ KB


In [74]:
#check for any nulls
print(df.isnull().sum())

Unnamed: 0        0
type              0
participant_id    0
event_id          0
created_at        0
object_id         0
dtype: int64


In [75]:
#check for any duplicate rows
df.duplicated().sum()

np.int64(0)

In [76]:
#drop unnecessary columns
df = df.drop(columns =['Unnamed: 0'])
df

,type,participant_id,event_id,created_at,object_id
0,question-answer,5bb21a6099567277d5856835,5bae4c0addbff83a25cef9e4,2018-10-04T09:03:57Z,5bb5bca6016c832eacc6a99f
1,question-answer,5b99018865b984370bf79d24,5b86edb9b60e0a2cb4447e78,2018-10-05T07:01:06Z,5b92b11c050fd04b90581398
2,question-answer,z112758121313,5b9e75a9416fae3713ad2eca,2018-10-09T12:26:25Z,5ba0d22631c09e2da13f27f8
3,question-answer,z211158485667,5ba8c5a723306e3a1808e753,2018-09-25T08:33:01Z,5ba8c6ebd0f1ca3a1e4b6e33
4,question-answer,5b96c5d5e3e3334b8a1f4fc1,5b9681ad52d9f12c9f277095,2018-09-10T19:44:14Z,5b96997e0f570e2c9e0a15c3
...,...,...,...,...,...
10095,question-answer,z261997914177,5bd6d4d19463065f1402e454,2018-11-15T15:53:25.163Z,5bd6d4d19463065f1402e517
10096,question-answer,z429552861175,5bc397b55ea8aa5f07340e14,2018-10-15T12:10:25.315Z,5bc4534019778f2cce061e35
10097,question-answer,z520001780226,5c025a203c6d2757c232e3ae,2018-12-07T10:33:14.823Z,5c025a203c6d2757c232e3e3
10098,question-answer,z255132820634,5b5736717d33bf57dd73434b,2018-09-19T09:18:35.797Z,5b5736717d33bf57dd734378


**Turning CSV data into parquet format partitoned by date**

In [77]:
df['created_at'] = pd.to_datetime(df['created_at'], format='ISO8601')
df['date'] = df['created_at'].dt.date.astype(str)

# For pandas to partition, we need to convert to a pyarrow table first
import pyarrow as pa
import pyarrow.parquet as pq

table = pa.Table.from_pandas(df, preserve_index=False)
pq.write_to_dataset(
    table,
    root_path='parquet_output',
    partition_cols=['date']
)

In [78]:
print(os.listdir("parquet_output"))

['date=2018-12-20', 'date=2018-11-27', 'date=2018-08-04', 'date=2018-12-05', 'date=2018-12-28', 'date=2018-08-20', 'date=2018-12-07', 'date=2018-09-15', 'date=2018-10-20', 'date=2018-12-26', 'date=2018-12-15', 'date=2018-11-19', 'date=2018-09-17', 'date=2018-08-26', 'date=2018-11-14', 'date=2018-10-04', 'date=2018-11-04', 'date=2018-12-04', 'date=2018-11-22', 'date=2018-07-30', 'date=2018-11-24', 'date=2018-09-05', 'date=2018-08-29', 'date=2018-08-06', 'date=2018-12-13', 'date=2018-08-28', 'date=2018-10-22', 'date=2018-10-16', 'date=2018-10-24', 'date=2018-09-28', 'date=2018-10-08', 'date=2018-11-23', 'date=2018-11-11', 'date=2018-09-27', 'date=2018-10-06', 'date=2018-11-01', 'date=2018-10-14', 'date=2018-09-19', 'date=2018-10-28', 'date=2018-10-25', 'date=2018-12-25', 'date=2018-11-18', 'date=2018-12-22', 'date=2018-09-04', 'date=2018-09-29', 'date=2018-09-23', 'date=2018-11-08', 'date=2018-10-30', 'date=2018-11-25', 'date=2018-11-15', 'date=2018-11-28', 'date=2018-12-12', 'date=2018-

In [79]:
df_check = pd.read_parquet("parquet_output")
print(df_check.head())
print(df_check.shape)

              type            participant_id                  event_id  \
0  question-answer             z182966690132  5b4cb63c16f6760aacc08e6c   
1  question-answer  5b5cb8beaf00183016a5d0ce  5b4cb63c16f6760aacc08e6c   
2  question-answer             z182966690132  5b4cb63c16f6760aacc08e6c   
3  question-answer  5b5cb8beaf00183016a5d0ce  5b4cb63c16f6760aacc08e6c   
4  question-answer             z424427928932  5b5e1649d5be9812181225bc   

                        created_at                 object_id        date  
0 2018-07-28 18:56:30.114000+00:00  5b55cd900c6372152bc85e93  2018-07-28  
1 2018-07-28 18:53:16.366000+00:00  5b5759f357c89657d1a884e0  2018-07-28  
2 2018-07-28 18:56:30.114000+00:00  5b55cd900c6372152bc85e93  2018-07-28  
3 2018-07-28 18:53:16.366000+00:00  5b5759f357c89657d1a884e0  2018-07-28  
4        2018-07-30 14:02:02+00:00  5b5e1ac800e5393010f48a57  2018-07-30  
(20200, 6)


In [80]:
print(pd.read_parquet("parquet_output").info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20200 entries, 0 to 20199
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   type            20200 non-null  object             
 1   participant_id  20200 non-null  object             
 2   event_id        20200 non-null  object             
 3   created_at      20200 non-null  datetime64[ns, UTC]
 4   object_id       20200 non-null  object             
 5   date            20200 non-null  category           
dtypes: category(1), datetime64[ns, UTC](1), object(4)
memory usage: 833.9+ KB
None


In [81]:
df_day = pd.read_parquet(
    "parquet_output",
    filters=[("date", "==", "2018-12-19")]
)

print(df_day.head())

              type                                     participant_id  \
0  question-answer  +726d637a576d5648584f66644b4f49503050526773394...   
1  question-answer                                     z1183900672622   
2  question-answer                           5ba8e02d7024ca77cdb62b00   
3  question-answer                                      z263752372834   
4  question-answer                                       z63775942563   

                   event_id                created_at  \
0  5c18da2e9c2e8d12f5606f02 2018-12-19 10:48:20+00:00   
1  5c08f565b0576f064ded0bbd 2018-12-19 08:56:57+00:00   
2  59d382fd1836dc1e3038ef06 2018-12-19 13:49:01+00:00   
3  5c196945214491124605c3df 2018-12-19 13:49:22+00:00   
4  5c19345e21d4ab122f3ae5c9 2018-12-19 13:03:00+00:00   

                  object_id        date  
0  5c18da2e9c2e8d12f5606f1a  2018-12-19  
1  5c17f9d871ce036960cd15cd  2018-12-19  
2  59d3efbb1836dc1e30399d13  2018-12-19  
3  5c196d6e1c4fa812441e7c24  2018-12-19  
4  5c193

**Make a graph to visualize the distribution of answers overtime**

In [82]:
df

,type,participant_id,event_id,created_at,object_id,date
0,question-answer,5bb21a6099567277d5856835,5bae4c0addbff83a25cef9e4,2018-10-04 09:03:57+00:00,5bb5bca6016c832eacc6a99f,2018-10-04
1,question-answer,5b99018865b984370bf79d24,5b86edb9b60e0a2cb4447e78,2018-10-05 07:01:06+00:00,5b92b11c050fd04b90581398,2018-10-05
2,question-answer,z112758121313,5b9e75a9416fae3713ad2eca,2018-10-09 12:26:25+00:00,5ba0d22631c09e2da13f27f8,2018-10-09
3,question-answer,z211158485667,5ba8c5a723306e3a1808e753,2018-09-25 08:33:01+00:00,5ba8c6ebd0f1ca3a1e4b6e33,2018-09-25
4,question-answer,5b96c5d5e3e3334b8a1f4fc1,5b9681ad52d9f12c9f277095,2018-09-10 19:44:14+00:00,5b96997e0f570e2c9e0a15c3,2018-09-10
...,...,...,...,...,...,...
10095,question-answer,z261997914177,5bd6d4d19463065f1402e454,2018-11-15 15:53:25.163000+00:00,5bd6d4d19463065f1402e517,2018-11-15
10096,question-answer,z429552861175,5bc397b55ea8aa5f07340e14,2018-10-15 12:10:25.315000+00:00,5bc4534019778f2cce061e35,2018-10-15
10097,question-answer,z520001780226,5c025a203c6d2757c232e3ae,2018-12-07 10:33:14.823000+00:00,5c025a203c6d2757c232e3e3,2018-12-07
10098,question-answer,z255132820634,5b5736717d33bf57dd73434b,2018-09-19 09:18:35.797000+00:00,5b5736717d33bf57dd734378,2018-09-19


In [96]:
#plot the yearly distribution of answers
df['date'] = df['created_at'].dt.date
daily_counts = df.groupby('date')['object_id'].count()

daily_counts_df = daily_counts.reset_index()
daily_counts_df.columns = ['Date', 'Answer_Count']

fig = px.area(
    daily_counts_df,
    x='Date',
    y='Answer_Count',
    title='<b>Yearly Distribution of Wooclap Answers 2018</b>',
    labels={'Answer_Count': 'Number of Answers'},
    template='plotly_white'
)

fig.update_layout(
title_font_size=22,
    xaxis_title="",
    yaxis_title="Total Answers Provided",
    hovermode="x unified",
    font_family="Arial",
    margin=dict(l=50, r=50, t=80, b=50)
)

fig.update_traces(
    line_color='#4c78a8',
    line_width=3,
    fillcolor='rgba(76, 120, 168, 0.2)'
)

fig.show()

In [84]:
#create a monthly answers distribution chart for September 2018
september_df = df[(df['created_at'].dt.year == 2018) & (df['created_at'].dt.month == 9)]
print(september_df)

                  type            participant_id                  event_id  \
3      question-answer             z211158485667  5ba8c5a723306e3a1808e753   
4      question-answer  5b96c5d5e3e3334b8a1f4fc1  5b9681ad52d9f12c9f277095   
8      question-answer             z971009355828  5b7191386323b230093388c9   
9      question-answer             z499375087320  5babf45adaf7f73a1d6379fa   
10     question-answer             z799834975535  5b8e6193b2f83013b504c0c7   
...                ...                       ...                       ...   
10079  question-answer  5ae18a6cf0e2bd60d85f2243  5b748d5f09166b12061dbeec   
10085  question-answer              z47872554836  5b7c049399042145c7078e04   
10089  question-answer              z64210511884  5ba3b775d45f482da4492b82   
10091  question-answer             z400637388822  5b97af98050fd04b90589ca8   
10098  question-answer             z255132820634  5b5736717d33bf57dd73434b   

                            created_at                 object_i

In [99]:
#create a monthly distribution of answers in September 2018
september_df['date'] = september_df['created_at'].dt.date
daily_counts = september_df.groupby('date')['object_id'].count().reset_index()
daily_counts.columns = ['Date', 'Answer_Count']

fig = px.area(
    daily_counts,
    x='Date',
    y='Answer_Count',
    title='<b>Monthly Distribution of Answers in September 2018</b>',
    labels={'Answer_Count': 'Number of Answers'},
    template='plotly_white'
)

fig.update_layout(
    title_font_size=22,
    xaxis_title="",
    yaxis_title="Total Answers Provided",
    hovermode="x unified",
    font_family="Arial",
    margin=dict(l=50, r=50, t=80, b=50)
)

fig.update_traces(
    line_color='#4c78a8',
    line_width=3,
    fillcolor='rgba(76, 120, 168, 0.2)'
)

fig.show()

**Proposing a methodology to identify different sessions in Wooclap**

1. Identify what columns/data should be utilized
    * In this case, I would use **created_at** (for date and time), **event_id** (to figure out if the same event was reused at a different day), and **participant_id** to see *who* and *how many* people were participating in EACH session

2. Convert created_at data into datetime format
    * df['created_at'] = pd.to_datetime(df['created_at'])
3. Group by participant_id, event_id, and created_id using the .ngroup() function in Pandas
    * df['session_id'] = df.groupby(['participant_id', 'event_id', 'created_at']).ngroup()
4. The final product will have the dataset labelled with a unique session ID column, helping us identify each **unique** session in Wooclap by date, event ID, and participant ID
